# Cumberland Pre-Development PEST First Slice

This notebook sets up a Cumberland calibration example around the same current GIS inputs used by the recent pre-development runs, but collapses the workflow to a single steady-state stress period for a lighter first PEST/pyEMU setup.

It follows the geometry/top-surface adjustments from `model_pit_pre_excavation.py`, then builds a one-period model with:

- `K` from `ks_v5.gpkg`
- drains from `drn.gpkg`
- steady recharge
- domain head targets built from calibration locations plus one pre-development target table row
- a `PestProject` with pilot-point `K`, drain elevation offsets, and drain conductance multipliers

In this first example, the head targets come from the earlier `cum9a_algo7h/cumb_obs.csv` model-output table, so they are pseudo-observations rather than true field measurements. The resulting template workspace is still useful for calibration plumbing and can be pointed at a real observation table later.

## Imports and package access

Load the standard Python utilities, GIS/data libraries, and the `myflopy` classes used to build the Cumberland pre-development model and its first-slice PEST workspace.

In [ ]:
from pathlib import Path
from datetime import datetime
import pickle
import re
import subprocess
import sys

import geopandas as gpd
import pandas as pd

project_root = Path.cwd().resolve()
src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import myflopy as mf
from myflopy import read_shp_gpkg
from myflopy.modflow.mf6.simulation import DisvGrid, TemporalDiscretization
from myflopy.modflow.mf6.simulation.packages import (
    Drains,
    InitialConditions,
    KFlow,
    OutputControl,
    Recharge,
    Storage,
)

## Workspace and input paths

Define a clean artifact workspace for this notebook run, then point to the live Cumberland GIS, mesh, and head files that the pre-development workflow already uses.

In [ ]:
artifact_root = Path(r"C:\Users\lukem\Python\Projects\myflopy\examples\mf6\artifacts")
run_family = "cumberland_predev_pest_first_slice"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
workspace_root = artifact_root / f"{run_family}_{timestamp}"
index = 1
while workspace_root.exists():
    workspace_root = artifact_root / f"{run_family}_{timestamp}_{index:02d}"
    index += 1
workspace_root.mkdir(parents=True)

model_workspace = workspace_root / "model"
pest_workspace = workspace_root / "pest"
latest_pointer = artifact_root / f"{run_family}_latest.txt"
latest_pointer.write_text(str(workspace_root), encoding="utf-8")

run_record = {
    "run_family": run_family,
    "workspace_root": str(workspace_root),
    "pest_workspace": str(pest_workspace),
    "created_at": timestamp,
}

print(f"Workspace root: {workspace_root}")
print(f"Model workspace parent: {model_workspace}")
print(f"PEST workspace: {pest_workspace}")
print(f"Latest-run pointer: {latest_pointer}")


## Load the existing mesh and adjust surfaces

Open the saved AlgoMesh-style Voronoi grid and starting heads, rebuild the active-domain mask, and apply the same pit, rejection, and local top-surface adjustments used in `model_pit_pre_excavation.py`.

In [ ]:
with open(iheads_path, "rb") as f:
    iheads = pickle.load(f)

with open(vor_algo, "rb") as f:
    vor = pickle.load(f)

vor.gdf_topbtm[0] = vor.gdf_topbtm[0].astype(float)
vor.gdf_topbtm[1] = vor.gdf_topbtm[1].astype(float)
reconciled = vor.reconcile_surfaces(min_sep=5, trigger_sep=5, which="top")
vor.gdf_topbtm[0] = reconciled[0].astype(float)
vor.gdf_topbtm[1] = reconciled[1].astype(float)

inactive_cells = vor.get_vor_cells_as_series(idomain_path).explode().tolist()
idomain = pd.Series(1 for _ in range(vor.ncpl))
idomain.iloc[inactive_cells] = 0

# Match the same pre-development surface adjustments used in model_pit_pre_excavation.py.
pit2_cells = vor.get_vor_cells_as_series(pit2)[0]
tb2 = vor.gdf_topbtm.loc[pit2_cells]
pit2_thickness = tb2.loc[:, 0] - tb2.loc[:, 1]
thin_cells = pit2_thickness[pit2_thickness < 25].index
vor.gdf_topbtm.loc[thin_cells, 0] = vor.gdf_topbtm.loc[thin_cells, 1] + 25

pit4_cells = vor.get_vor_cells_as_series(pit4)[0]
tb4 = vor.gdf_topbtm.loc[pit4_cells]
pit4_thickness = tb4.loc[:, 0] - tb4.loc[:, 1]
thin_cells = pit4_thickness[pit4_thickness < 20].index
vor.gdf_topbtm.loc[thin_cells, 0] = vor.gdf_topbtm.loc[thin_cells, 1] + 20

rej_cells = vor.get_vor_cells_as_series(rej, name_field="name")
rej_df = pd.concat([read_shp_gpkg(rej).set_index("name"), rej_cells], axis=1)
for name, row in rej_df.iterrows():
    cells_to_check = vor.gdf_topbtm.loc[row.cells, 0]
    low_cells = cells_to_check.loc[cells_to_check < row.thickness].index.to_list()
    vor.gdf_topbtm.loc[low_cells, 0] = row.thickness

liz_cells = vor.get_vor_cells_as_series(read_shp_gpkg(mfr_polys).loc[8, "geometry"])[0]
lizmfr_cells = [cell for cell in liz_cells if cell not in inactive_cells]
vor.gdf_topbtm.loc[lizmfr_cells, 0] += 10

reconciled = vor.reconcile_surfaces(min_sep=5, trigger_sep=5, which="top")
vor.gdf_topbtm[0] = reconciled[0].astype(float)
vor.gdf_topbtm[1] = reconciled[1].astype(float)

top = vor.gdf_topbtm[0].values
botm = vor.gdf_topbtm[1].values

vor.ncpl, int(idomain.sum())

## Build and run the one-period steady-state model

Create a simplified pre-development Cumberland model with one steady-state stress period, then attach `DISV`, `K`, initial heads, storage, recharge, and drains using the current production GIS inputs.

In [ ]:
model = mf.SimulationBase(
    name="cum_predev_ss",
    nper=1,
    vor=vor,
    mf_folder_path=model_workspace,
)

DisvGrid(
    vor=vor,
    model=model,
    nlay=1,
    top=top,
    bottom=botm,
    idomain=idomain,
)
TemporalDiscretization(model=model, period_data=[[1.0, 1, 1.0]])
OutputControl(model=model)

k_array = mf.KFromVector(
    model=model,
    vor=vor,
    shp_gpkg=ks_v5,
    uid="name",
    crs=2926,
    idomain_path=idomain_path,
).from_vector(nlay=1, defaults=[10.0])
KFlow(model=model, k=k_array[0], k33_vert=(k_array[0] / 10.0), save_specific_discharge=False)

InitialConditions(model=model, vor=vor, strt=[iheads])
Storage(
    model=model,
    specific_yield=0.2,
    specific_storage=1e-4,
    sto_steady={0: True},
    sto_transient={},
)

rch_dict = {
    0: [[(0, int(cell)), 0.007] for cell in range(vor.ncpl) if idomain.iloc[cell] == 1]
}
Recharge(model=model, vor=vor, rch_dict=rch_dict)

drn_fields = {
    "name": "name",
    "height_over_btm": "height",
    "conductance": "cond",
    "layer": "layer",
    "min_elev": "min_elev",
}
drn_spd = mf.DRNFromVector(
    model=model,
    vor=vor,
    shp_gpkg=drn_path,
    uid="name",
    crs=2926,
    idomain_path=idomain_path,
).from_vector(fields=drn_fields, edges_only=False, top_drain=True)
Drains(model=model, stress_period_data=drn_spd)

success, _ = model.run_simulation()
success

## Assemble head targets

Build a `HeadTargets` object from the Cumberland calibration locations and a single-row target table. In this notebook these are pseudo-observations from an earlier model output file, which is useful for verifying that PEST++ can recover a known solution.

In [ ]:
locs = gpd.read_file(calib_locs)
obs = pd.read_csv(predev_obs)

obs_names = [name for name in obs.columns if name != "time" and name in set(locs["ExploName"].astype(str))]
locs = locs[locs["ExploName"].astype(str).isin(obs_names)].copy()
locs["name"] = locs["ExploName"].astype(str)
locs["layer_num"] = 0
locs["weight"] = 1.0

print("Using pseudo-target heads from prior model output:", predev_obs)

values = obs.loc[[0], ["time"] + obs_names].copy().rename(columns={"time": "per"})
values["per"] = 0

targets = mf.HeadTargets(
    locations=locs[["name", "layer_num", "weight", "geometry"]],
    values=values,
    name_column="name",
    layer_column="layer_num",
    time_column="per",
)

targets.summary()

## Build the PEST workspace

Create a parameter-source copy of the drain GIS with safe parameter names, then configure a `PestProject` with pilot-point `K`, drain elevation offsets, drain conductance multipliers, and the head targets.

In [ ]:
raw_drn = gpd.read_file(drn_path)
raw_drn["par_name"] = raw_drn["name"].astype(str).str.lower().map(
    lambda s: re.sub(r"[^a-z0-9]+", "_", s).strip("_")
)
param_drn = workspace_root / "drn_param_source.gpkg"
if param_drn.exists():
    param_drn.unlink()
raw_drn.to_file(param_drn, driver="GPKG")

pest = mf.PestProject(
    model=model,
    name="cum_predev_ss_pest",
    workspace=pest_workspace,
    start_datetime="2024-01-01",
)

pest.add_parameter(
    mf.KPilotPointParameter(
        name="hk",
        source=mf.VectorParameterSource(
            path=ks_v5,
            value_column="k",
            feature_id_column="name",
            zone_column="name",
            layer_column="layer",
            crs=2926,
        ),
        bounds=(0.25, 4.0),
        bounds_mode="multiplier",
        transform="log",
        pp_spacing=2500.0,
        geostruct=mf.ExpGeoStruct(range=4000.0, transform="log"),
    )
)

pest.add_parameter(
    mf.DrainElevationParameter(
        name="drn_elev",
        source=mf.VectorParameterSource(
            path=param_drn,
            value_column="height",
            feature_id_column="par_name",
            layer_column="layer",
            crs=2926,
        ),
        bounds=(-10.0, 10.0),
        bounds_mode="absolute",
    )
)

pest.add_parameter(
    mf.DrainConductanceParameter(
        name="drn_cond",
        source=mf.VectorParameterSource(
            path=param_drn,
            value_column="cond",
            feature_id_column="par_name",
            layer_column="layer",
            crs=2926,
        ),
        bounds=(0.25, 4.0),
        bounds_mode="multiplier",
        transform="log",
    )
)

pest.add_observation(mf.HeadTargetObservationSpec(targets=targets))
pst = pest.build_pst("cum_predev_ss.pst")

pst.npar_adj, pst.nobs

## Inspect the generated template files

Take a quick look at the key CSV, template, control, and helper files that were written into the PEST workspace.

In [ ]:
sorted(path.name for path in pest_workspace.iterdir() if path.suffix in {".csv", ".tpl", ".json", ".pst", ".py"})[:20]

## Optional forward-run smoke check

Perturb the pilot-point and drain parameter CSVs, rerun the generated `forward_run.py`, and confirm that the simulated heads respond. This is a quick validation that the template workspace is genuinely wired to the MF6 inputs.

In [ ]:
# Optional smoke check: perturb the template inputs and rerun the generated forward model.
initial_heads = pd.read_csv(pest_workspace / "hds_simulated_heads.csv")

hk_frame = pd.read_csv(pest_workspace / "hk_pilot_points.csv")
hk_frame["value"] = hk_frame["value"] * 1.05
hk_frame.to_csv(pest_workspace / "hk_pilot_points.csv", index=False)

drn_elev_frame = pd.read_csv(pest_workspace / "drn_elev_drain_elevation.csv")
drn_elev_frame["value"] = 0.1
drn_elev_frame.to_csv(pest_workspace / "drn_elev_drain_elevation.csv", index=False)

drn_cond_frame = pd.read_csv(pest_workspace / "drn_cond_drain_conductance.csv")
drn_cond_frame["value"] = 0.95
drn_cond_frame.to_csv(pest_workspace / "drn_cond_drain_conductance.csv", index=False)

subprocess.run([sys.executable, "forward_run.py"], cwd=pest_workspace, check=True)
rerun_heads = pd.read_csv(pest_workspace / "hds_simulated_heads.csv")

comparison = initial_heads.merge(rerun_heads, on=["per"], suffixes=("_base", "_perturbed"))
obs_columns = [col for col in initial_heads.columns if col != "per"]
long_rows = []
for name in obs_columns:
    long_rows.append(
        {
            "name": name,
            "sim_head_base": comparison.loc[0, f"{name}_base"],
            "sim_head_perturbed": comparison.loc[0, f"{name}_perturbed"],
        }
    )
comparison_long = pd.DataFrame(long_rows)
comparison_long["delta"] = comparison_long["sim_head_perturbed"] - comparison_long["sim_head_base"]
comparison_long.sort_values("delta", key=lambda s: s.abs(), ascending=False).head()